In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CrossAttentionHead(nn.Module):
    def __init__(self, n_embd, head_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, context):
        B, T_x, C = x.shape
        B, T_c, C_c = context.shape
        q = self.query(x)
        k = self.key(context)
        v = self.value(context)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

In [14]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([CrossAttentionHead(n_embd, head_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, context):
        out = torch.cat([h(x, context) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

PART C

In [15]:
import os
import ssl
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

ssl._create_default_https_context = ssl._create_unverified_context
opener = urllib.request.build_opener()
opener.addheaders = [('User-agent', 'Mozilla/5.0')]
urllib.request.install_opener(opener)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
class CausalHead(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, head_size, block_size, dropout=0.1):
        super().__init__()
        self.heads = nn.ModuleList([CausalHead(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_head * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [17]:
class BidirectionalHead(nn.Module):
    def __init__(self, n_embd, head_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = F.softmax(wei, dim=-1)
        out = self.dropout(wei) @ v
        return out

class MultiHeadBidirectionalAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([BidirectionalHead(n_embd, head_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class BlockBidirectional(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        self.sa = MultiHeadBidirectionalAttention(n_embd, n_head, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class ViTEncoder(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, n_embd=128, n_head=4, n_layer=2, dropout=0.1):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(in_chans, n_embd, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, n_embd))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, n_embd))
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([BlockBidirectional(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.norm = nn.LayerNorm(n_embd)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed
        x = self.dropout(x)
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)

In [18]:
class CrossAttentionHeadCustom(nn.Module):
    def __init__(self, n_embd, head_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, context, store_attn=False):
        q = self.query(x)
        k = self.key(context)
        v = self.value(context)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = F.softmax(wei, dim=-1)
        if store_attn:
            self.last_attn_weights = wei.detach()
        out = self.dropout(wei) @ v
        return out

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, head_size, dropout=0.1):
        super().__init__()
        self.heads = nn.ModuleList([CrossAttentionHeadCustom(n_embd, head_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_head * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, context, store_attn=False):
        out = torch.cat([h(x, context, store_attn=store_attn) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        head_size = n_embd // n_head
        self.self_attn = MultiHeadAttention(n_embd, n_head, head_size, block_size, dropout)
        self.cross_attn = MultiHeadCrossAttention(n_embd, n_head, head_size, dropout)
        self.ffn = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, image_features, store_attn=False):
        x = x + self.self_attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), image_features, store_attn=store_attn)
        x = x + self.ffn(self.ln3(x))
        return x

In [19]:
class MultimodalModel(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=2, block_size=64, img_size=32, patch_size=4):
        super().__init__()
        self.vision_encoder = ViTEncoder(img_size=img_size, patch_size=patch_size, n_embd=n_embd, n_head=n_head, n_layer=2)
        self.token_embed = nn.Embedding(vocab_size, n_embd)
        self.pos_embed = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([DecoderBlock(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_final = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.block_size = block_size

    def forward(self, image, text_ids, targets=None, store_attn=False):
        image_feats = self.vision_encoder(image)
        B, T = text_ids.shape
        tok = self.token_embed(text_ids)
        pos = self.pos_embed(torch.arange(T, device=text_ids.device))
        x = tok + pos
        for blk in self.blocks:
            x = blk(x, image_feats, store_attn=store_attn)
        x = self.ln_final(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

In [20]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)

classes = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

unique_chars = sorted(list(set("".join([f"this is a {c}." for c in classes]))))
stoi = {ch: i for i, ch in enumerate(unique_chars)}
itos = {i: ch for i, ch in enumerate(unique_chars)}
vocab_size = len(unique_chars)

model = MultimodalModel(vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

loss_history = []
steps =10
data_iter = iter(train_loader)

for step in range(steps):
    try:
        images, labels = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        images, labels = next(data_iter)
        
    images = images.to(device)
    batch_captions = [f"this is a {classes[l]}.".ljust(21) for l in labels]
    
    encoded_batch = []
    for cap in batch_captions:
        encoded_batch.append([stoi[c] for c in cap])
        
    text_tensor = torch.tensor(encoded_batch).to(device)
    
    input_ids = text_tensor[:, :-1]
    target_ids = text_tensor[:, 1:]
    
    optimizer.zero_grad()
    logits, loss = model(images, input_ids, target_ids)
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    if step % 20 == 0:
        print(f"Step {step:03d} | Multimodal Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Multimodal Convergence Tracking")
plt.grid(True)
plt.savefig("training_curve.png")
plt.close()

Step 000 | Multimodal Loss: 3.1295


In [21]:
model.eval()
sample_img, sample_lbl = train_dataset[0]
sample_img = sample_img.unsqueeze(0).to(device)
true_class = classes[sample_lbl]

generated_ids = [stoi['t']]
for _ in range(30):
    input_tensor = torch.tensor([generated_ids]).to(device)
    with torch.no_grad():
        logits, _ = model(sample_img, input_tensor, store_attn=True)
    next_id = logits[0, -1].argmax().item()
    generated_ids.append(next_id)
    if itos[next_id] == '.':
        break

generated_caption = "".join([itos[i] for i in generated_ids])

with open("samples.txt", "w") as f:
    f.write(f"Target Image True Category: {true_class}\n")
    f.write(f"Model Generated Autoregressive Text: {generated_caption}\n")

print("--- Text Output Generated successfully inside samples.txt ---")

attn_layer = model.blocks[0].cross_attn.heads[0]
attention_map = attn_layer.last_attn_weights[0].cpu().numpy()

plt.figure(figsize=(10, 6))
plt.imshow(attention_map, aspect='auto', cmap='viridis')
plt.xlabel("Vision Context Token Indexes (Patches)")
plt.ylabel("Text Query Step Sequence")
plt.yticks(range(len(generated_ids[:-1])), [itos[i] for i in generated_ids[:-1]])
plt.title("Cross-Attention Alignment Heatmap")
plt.colorbar()
plt.savefig("attention_viz.png")
plt.close()
print("--- Deliverable figures saved safely to disk ---")

--- Text Output Generated successfully inside samples.txt ---
--- Deliverable figures saved safely to disk ---
